In [ ]:
# Note: this script requires the program FFMPEG to be installed and accessible from the command line.
# FFMPEG can be downloaded from https://ffmpeg.org/download.html
# Timestamps in the subtitle files are written based on the date_modified of the file, its framerate (extracted from the video) and the photos/second taken for the timelapse (user-supplied)
# If the camera date-time was off, and therefore the modification date, you can use the program XNviewMP to batch-correct the timestamp (select files, use Metadata -> Change Timestamp)

# The script contains five sections:
# 1. Function definitions
# 2. Compress a single video (example)
# 3. Combine two videos into a single compressed video (example)
# 4. Run a single job using the defined function. Combine all videos from an input folder into a single compressed video with timestamp subtitles. Crop the start and end if desired.
# 5. Batch processing: run multiple jobs in parallel, from a excel list of input folders, output file names etc.

# Note: to write correct timestamp subtitles, remember to set how many photos per second were taken for the timelapse videos, using variable sub_step_real_s
 
from pathlib import Path
from datetime import datetime, timezone, timedelta
import cv2 # requires package opencv. Use e.g. conda install -c conda-forge opencv
import pandas as pd
import math, subprocess, json, warnings

In [ ]:
# 1) Define functions
folder_ffmpeg = r"C:\Program Files\EibolSoft\FFmpeg Batch AV Converter"
folder_ffprobe = r'C:\Users\dpoppema\Downloads' # Probably in the same folder as ffmpeg normally, separate for me

ffmpeg_exe  = str(Path(folder_ffmpeg) / "ffmpeg.exe")
ffprobe_exe = str(Path(folder_ffprobe) / "ffprobe.exe")

# Helper functions ---------------------------------------------------------------------------------
def get_video_info(video_path):
    """Returns (mtime, fps, frames, dt_sec_video, dt_sec_real, t_start)"""
    p = Path(video_path)
    mtime = datetime.fromtimestamp(p.stat().st_mtime)
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    dt_sec_video = frames / fps if fps and fps > 0 else 0.0
    cap.release()
    
    dt_sec_real = dt_sec_video * fps / 2  # adjust for timelapse
    t_start = mtime - timedelta(seconds=dt_sec_real)
    
    return mtime, fps, frames, dt_sec_video, dt_sec_real, t_start

def srt_timestamp(seconds):
    ms_total = int(round( (seconds) * 1000))
    h, rem = divmod(ms_total, 3600_000)
    m, rem = divmod(rem, 60_000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def probe_duration(path):
    p = subprocess.run(
        [ffprobe_exe, "-v", "error", "-show_entries", "format=duration", "-of", "json", str(path)],
        capture_output=True, text=True, check=True
    )
    return float(json.loads(p.stdout)["format"]["duration"])

# Main function -----------------------------------------------------------------------------------
def process_job(folder_in,
                folder_out,
                filename_out,
                video_title,
                start_trim,
                end_trim,
                sub_step_real_s: float = 2.0,
                video_pattern: str = "*.mp4"):
    """
    Process a single job: build merged subtitles, concat, trim, and re-encode with soft subtitles.
    - folder_in: folder containing input video files (sorted by name)
    - folder_out: output folder
    - filename_out: output file name (e.g., 'merged_trimmed_sub.mkv')
    - video_title: title to display (top-left) in SRT using {\an7}
    - start_trim: seconds to cut from start of combined stream
    - end_trim: seconds to cut from end of combined stream
    - sub_step_real_s: real-time interval between consecutive subtitle cues
    - video_pattern: glob for input files (default '*.mp4')
    """
    folder_in = Path(folder_in)
    out_path = Path(folder_out) / filename_out
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # If the output already exists: warn and skip this job
    if out_path.exists():
        warnings.warn(f"Output already exists, skipping: {out_path}")
        return

    # Collect videos
    video_files = sorted(folder_in.glob(video_pattern))
    if not video_files:
        raise RuntimeError(f"No videos matching {video_pattern} in {folder_in}")

    # Compute total duration and output duration after trims
    dur_total = sum(probe_duration(vf) for vf in video_files)
    out_len = dur_total - (float(start_trim) + float(end_trim))
    if out_len <= 0:
        raise ValueError(f"Trim too large: total={dur_total:.3f}s -> output={out_len:.3f}s")

    # Write the SRT in the INPUT folder (temporary) and remove it after success
    srt_path = folder_in / "merged_subtitles.srt"
    # if srt_parth already exists, remove it
    if srt_path.exists():
        try:
            srt_path.unlink()
        except Exception:
            pass

    cue_index = 1
    video_offset = 0.0  # cumulative video time offset across files

    with srt_path.open("w", encoding="utf-8") as f:
        for video_file in video_files:
            mtime, fps, frames, dt_sec_video, dt_sec_real, t_start = get_video_info(video_file)
            ratio = (dt_sec_real / dt_sec_video) if dt_sec_video and dt_sec_video > 0 else 0.0

            def real_time_at(video_seconds: float):
                return t_start + timedelta(seconds=video_seconds * ratio)

            step_video_s = (sub_step_real_s / ratio) if ratio > 0 else 1.0
            n_cues = max(1, math.ceil(dt_sec_video / step_video_s))

            for i in range(n_cues):
                t0_v = i * step_video_s
                t1_v = min((i + 1) * step_video_s, dt_sec_video)

                # Offset by cumulative video time from previous files
                t0_merged = video_offset + t0_v
                t1_merged = video_offset + t1_v

                text = real_time_at(t0_v).strftime("%Y-%m-%d %H:%M:%S")

                # Title at top (ASS-style position tag; works in many players when muxed as SRT in MKV)
                f.write(f"{cue_index}\n")
                f.write(f"{srt_timestamp(t0_merged)} --> {srt_timestamp(t1_merged)}\n")
                f.write(f"{{\\an7}} {video_title}\n\n")

                # Timestamp at bottom
                f.write(f"{cue_index + 1}\n")
                f.write(f"{srt_timestamp(t0_merged)} --> {srt_timestamp(t1_merged)}\n")
                f.write(f"{text}\n\n")

                cue_index += 2

            video_offset += dt_sec_video

    # Concat list for ffmpeg
    concat_file = folder_in / "concat_list.txt"
    # if concat_file already exists, remove it
    if concat_file.exists():
        try:
            concat_file.unlink()
        except Exception:
            pass

    with concat_file.open("w", encoding="utf-8") as f:
        for vf in video_files:
            f.write(f"file '{vf.as_posix()}'\n")

    # Build ffmpeg command: concat -> trim -> re-encode -> add subtitles with timing shifted by start_trim
    # to rotate video 90°, add "-vf", "transpose=1" after the -map lines. For 180°, add "-vf", "transpose=2,transpose=2",
    cmd = [
        ffmpeg_exe,
        "-f", "concat", "-safe", "0",
        "-ss", f"{start_trim}",        # trim next input, i.e. video
        "-i", str(concat_file),        # input: all video files
        "-ss", f"{start_trim}",        # trim next input, i.e. subtitle
        "-i", srt_path,                # subtitle input
        "-t", f"{out_len}",                     # keep this duration
        "-map", "0:v:0",                        # video from concat
        "-map", "1:0",                          # subtitles from srt
        # Video encode (HEVC QuickSync)
        "-c:v", "hevc_qsv",
        "-preset", "medium",
        "-global_quality", "26",
        "-look_ahead", "1",
        "-r", "30000/1001",
        # Subtitles in MKV (soft subs)
        "-c:s", "srt",
        "-metadata:s:s:0", "language=eng",
        "-disposition:s:0", "default+forced",
        str(out_path)
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)

    # Cleanup temp concat list
    try:
        concat_file.unlink()
    except Exception:
        pass

    if res.returncode == 0:
        # Print once per job, after successful mux
        print(f"Success: {out_path}")
        # Remove the temporary SRT created in the input folder
        try:
            srt_path.unlink()
        except Exception:
            pass
    else:
        print("ffmpeg error:\n", res.stderr)
        raise RuntimeError("ffmpeg failed")

In [ ]:
# 2) SIMPLY ENCODE A SINGLE VIDEO, reduce file size
file_in = r'O:\test\in_short\GX010001.mp4'
file_out = r'O:\test\out\GX010001.mp4'
folder_ffmpeg = r'C:\Program Files\EibolSoft\FFmpeg Batch AV Converter'

# Ensure output folder exists
Path(file_out).parent.mkdir(parents=True, exist_ok=True)

# Path to ffmpeg executable
ffmpeg_exe = Path(folder_ffmpeg) / "ffmpeg.exe"

# Build command
cmd = [
    str(ffmpeg_exe),
    "-i", file_in,
    "-c:v", "hevc_qsv",       # Codec: HEVC with Intel QuickSync acceleration. Change if you don't have an Intel processor with quicksync. 
    "-preset", "medium",
    "-global_quality", "26",  # higher number is higher compression, lower quality. Range is 1 to 51. 
    "-look_ahead", "1",
    "-r", "30000/1001",       # Framerate. Set here to maintain the original framerate of 29.97 fps
    file_out
]

# Run ffmpeg
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Success! Video saved to: {file_out}")
else:
    print("Error during encoding:")
    print(result.stderr)

In [ ]:
# 3) COMBINE TWO VIDEOS IN A LONGER VIDEO, WITH COMPRESSION
# This is usefull for GoPro videos that are split in multiple files due to size limits
file_in1 = r'O:\test\in_short\GX019761.mp4'
file_in2 = r'O:\test\in_short\GX019765.mp4'
file_out = r'O:\test\out\merged2.mp4'
folder_ffmpeg = r'C:\Program Files\EibolSoft\FFmpeg Batch AV Converter'

# Ensure output folder exists
Path(file_out).parent.mkdir(parents=True, exist_ok=True)

# Path to ffmpeg executable
ffmpeg_exe = Path(folder_ffmpeg) / "ffmpeg.exe"

# Create a temporary file list for ffmpeg concat
concat_file = Path(file_out).parent / "concat_list.txt"
with concat_file.open("w") as f:
    f.write(f"file '{Path(file_in1).as_posix()}'\n")
    f.write(f"file '{Path(file_in2).as_posix()}'\n")

# Build command using concat demuxer (fast, no re-encoding if same codec)
cmd = [
    str(ffmpeg_exe),
    "-f", "concat",
    "-safe", "0",
    "-i", str(concat_file),
    "-c:v", "hevc_qsv",       # Codec: HEVC with Intel QuickSync acceleration. Change if you don't have an Intel processor with quicksync. 
    "-preset", "medium",
    "-global_quality", "26",  # higher number is higher compression, lower quality. Range is 1 to 51. 
    "-look_ahead", "1",
    "-r", "30000/1001",       # Framerate. Set here to maintain the original framerate of 29.97 fps
    file_out
]

# Run ffmpeg
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

# Clean up temporary file
concat_file.unlink()

if result.returncode == 0:
    print(f"Success! Merged video saved to: {file_out}")
else:
    print("Error during merging:")
    print(result.stderr)

In [ ]:
# 4) USE THE FUNCTION DEFINED BEFORE: MERGE AND COMPRESS ALL VIDEOS IN A FOLDER, CROP START AND END TIME, ADD SUBTITLE WITH TIMESTAMPS

# Define input parameters
sub_step_real_s = 2.0                                                                   # display a new subtitle every n seconds real time. With timelapse of 10x speed, this is every 0.2s video time
folder_in       = r'E:\2024-12-18 to 2024-12-20, Storm 1\GoPros\GoPro1\high tide 2'
folder_out      = r'O:\HybridDune experiment\GoPros\Storm 1'
filename_out    = 'HybridDune timelapse - S2, storm 1, high tide 2.mkv'                 # NB: output format must be .mkv
video_title     = 'S2 Sandy Dune, Storm 1'                                              # title to show in top-left of video                                 
start_trim      = 10.0                                                                  # seconds to trim from start of combined video
end_trim        = 10.0                                                                  # seconds to trim from end of combined video

# Call function
process_job(folder_in, folder_out, filename_out, video_title, start_trim, end_trim, sub_step_real_s=sub_step_real_s)


In [ ]:
# 5) BATCH PROCESSING: RUN MULTIPE JOBS, FROM A LIST OF INPUT FOLDERS, OUTPUT FILE NAMES ETC. 
# This section runs two jobs concurrently using the existing process_job function. Lists with input folders etc are read from an excel file. 

# Safety checks: validates that input folders are unique and exist, and that output filenames are unique.
# Note: this script writes two temporary files in each input folder: the subtitle, and list of videos. Therefore,
# input folders must be unique across jobs. If not, run jobs sequentially.

# Driver: read Excel and process all rows
# Expected columns by position (1-based in description):
# 1: folder_in, 2: folder_out, 3: filename_out, 4: video_title, 5: start_trim, 6: end_trim
# The first line in the excel file is assumed to be a header and is ignored.
# The filename (column 3) should be without extension, an .mkv extenstion is added automatically

file_overview = r'O:\test\GoPro_overrview_2.xlsx'
sub_step_real_s = 2.0
df = pd.read_excel(file_overview, header=0)
print(f"Loaded overview with {len(df)} rows from {file_overview}")

from concurrent.futures import ThreadPoolExecutor, as_completed

# Loop over all rows in df, processing two at a time

# Collect job parameters
jobs = []
for j in range(len(df)):
    row = df.iloc[j]
    folder_in  = row.iloc[0]
    folder_out = row.iloc[1]
    filename_out = str(row.iloc[2])
    if not filename_out.lower().endswith('.mkv'):
        filename_out += '.mkv'
    video_title  = row.iloc[3]
    start_trim   = float(row.iloc[4])
    end_trim     = float(row.iloc[5])
    
    jobs.append({
        'index': j,
        'folder_in': folder_in,
        'folder_out': folder_out,
        'filename_out': filename_out,
        'video_title': video_title,
        'start_trim': start_trim,
        'end_trim': end_trim
    })

# Validation A: Check that all input folders are unique
input_folders = [job['folder_in'] for job in jobs]
if len(input_folders) != len(set(input_folders)):
    raise ValueError(f"Input folders must be unique for parallel execution: {input_folders}")

# Validation B: Check that all input folders exist
for job in jobs:
    folder_path = Path(job['folder_in'])
    if not folder_path.exists():
        raise FileNotFoundError(f"Input folder does not exist: {folder_path}")

# Validation C: Check that all output filenames (full paths) are unique
output_paths = [str(Path(job['folder_out']) / job['filename_out']) for job in jobs]
if len(output_paths) != len(set(output_paths)):
    raise ValueError(f"Output file paths must be unique for parallel execution: {output_paths}")

print("✓ All validations passed")
print(f"Processing {len(jobs)} jobs with max 2 concurrent...")

# Run jobs in parallel using existing process_job function (max 2 at a time)
with ThreadPoolExecutor(max_workers=2) as ex:
    futures = {ex.submit(
        process_job,
        job['folder_in'],
        job['folder_out'],
        job['filename_out'],
        job['video_title'],
        job['start_trim'],
        job['end_trim'],
        sub_step_real_s
    ): job for job in jobs}
    
    # Process results as they complete
    for fut in as_completed(futures):
        job = futures[fut]
        try:
            fut.result()  # process_job prints its own success/failure messages
            print(f"✓ Job {job['index']+1} ({job['filename_out']}) completed")
        except Exception as e:
            print(f"✗ Job {job['index']+1} ({job['filename_out']}) failed: {e}")

print("\nAll parallel jobs finished")
